In [1]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from river import linear_model, optim, preprocessing
# from river.compose import pure_inference_mode

In [2]:
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom)


def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan

    return 100.0 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

In [3]:
def build_model():
    model = (
        preprocessing.StandardScaler()
        | linear_model.LinearRegression(
            optimizer=optim.Adam(0.01),
            l2=1e-4,
            intercept_lr=0.01,
        )
    )
    return model

In [4]:
train = pd.read_parquet("data/gold/train.parquet").copy()
val = pd.read_parquet("data/gold/val.parquet").copy()

y_col = "Sum of кВт"

for df in [train, val]:
    df["ts"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"],
            hour=df["Hour"] - 1,  # only keep this if Hour is 1..24
        ),
        errors="coerce"
    )

train = train.sort_values("ts").reset_index(drop=True)
val = val.sort_values("ts").reset_index(drop=True)

# Drop rows with invalid timestamps or missing target
train = train.dropna(subset=["ts", y_col]).reset_index(drop=True)
val = val.dropna(subset=["ts", y_col]).reset_index(drop=True)

# Features = everything except target and timestamp
exclude_cols = {y_col, "ts"}
feature_cols = [c for c in train.columns if c not in exclude_cols]

print("Train shape:", train.shape)
print("Val shape:", val.shape)
print("Number of features:", len(feature_cols))
print("First val timestamp:", val["ts"].min())
print("Last val timestamp:", val["ts"].max())

Train shape: (4864324, 633)
Val shape: (304499, 633)
Number of features: 631
First val timestamp: 2025-06-30 00:00:00
Last val timestamp: 2025-07-31 22:00:00


In [ ]:
model = build_model()

train_subset = train[feature_cols + [y_col]]

for row in tqdm(
    train_subset.itertuples(index=False, name=None),
    total=len(train_subset),
    desc="Initial training"
):
    x_vals = row[:-1]
    y = row[-1]
    x = dict(zip(feature_cols, x_vals))
    model.learn_one(x, y)

Initial training:   0%|          | 0/4864324 [00:00<?, ?it/s]

In [ ]:
val = val.copy()
val["date"] = val["ts"].dt.date

daily_results = []
all_preds = []

unique_days = sorted(val["date"].unique())

for day in tqdm(unique_days, desc="Day-ahead validation"):
    day_df = val[val["date"] == day].sort_values("ts")
    day_subset = day_df[["ts", "date"] + feature_cols + [y_col]]

    y_true_day = []
    y_pred_day = []

    # Predict whole day first
    for row in day_subset.itertuples(index=False, name=None):
        ts = row[0]
        date_val = row[1]
        x_vals = row[2:-1]
        y_true = row[-1]

        x = dict(zip(feature_cols, x_vals))
        y_pred = model.predict_one(x)

        if y_pred is None:
            y_pred = 0.0

        y_true_day.append(y_true)
        y_pred_day.append(y_pred)

        all_preds.append({
            "ts": ts,
            "date": date_val,
            "y_true": y_true,
            "y_pred": y_pred,
        })

    # Daily metrics
    day_mape = mape(y_true_day, y_pred_day)
    day_smape = smape(y_true_day, y_pred_day)

    daily_results.append({
        "date": day,
        "n_obs": len(day_df),
        "MAPE": day_mape,
        "SMAPE": day_smape,
        "actual_sum": float(np.sum(y_true_day)),
        "pred_sum": float(np.sum(y_pred_day)),
        "bias_sum": float(np.sum(y_pred_day) - np.sum(y_true_day)),
    })

    # Then learn on that day's true values
    for row in day_subset.itertuples(index=False, name=None):
        x_vals = row[2:-1]
        y = row[-1]
        x = dict(zip(feature_cols, x_vals))
        model.learn_one(x, y)

In [ ]:
daily_metrics = pd.DataFrame(daily_results)
preds_df = pd.DataFrame(all_preds)

# Average of daily metrics
avg_daily_mape = daily_metrics["MAPE"].mean()
avg_daily_smape = daily_metrics["SMAPE"].mean()

# Global metrics across all validation rows
global_mape = mape(preds_df["y_true"], preds_df["y_pred"])
global_smape = smape(preds_df["y_true"], preds_df["y_pred"])

print(f"Average daily MAPE over val:  {avg_daily_mape:.4f}%")
print(f"Average daily SMAPE over val: {avg_daily_smape:.4f}%")
print()
print(f"Global MAPE over all val rows:  {global_mape:.4f}%")
print(f"Global SMAPE over all val rows: {global_smape:.4f}%")

In [ ]:
daily_metrics.head()

In [ ]:
preds_df.head()